<span style='font-size:50px; font-weight:bold;'>Recommendation System</span>

---

In [1]:
import pandas as pd
import numpy as np

<span style='font-size:22px; font-weight:500'>
    ➤ Load Data:
</span>

In [2]:
df = pd.read_csv('data/IMDb_movies_cleaned.csv')
df.head()

,title,genres,description,keywords,release_date,duration,content_rating,director,star_1,star_2,...,user_reviews,critic_reviews,watchlist_count,gross,budget,movie_url,category,currency_type,gross_usd,budget_usd
0,Kate & Leopold,Comedy|Fantasy|Romance,An English Duke from 1876 is inadvertently dra...,"time travel,time travel romance,female time tr...",2001-12-25,118,PG-13,James Mangold,Meg Ryan,Hugh Jackman,...,376.0,92.0,66600.0,"$76,019,048","$48,000,000 (estimated)",https://www.imdb.com/title/tt0035423/,Mid Rated,$,76019048.0,48000000.0
1,Ko to tamo peva,Adventure|Comedy|Drama,"It&apos;s April 5, 1941, somewhere in Serbia. ...","road trip,sex scene,female nudity,year 1941,yu...",1980-06-30,86,Unknown,Slobodan Sijan,Pavle Vuisic,Dragan Nikolic,...,42.0,11.0,12200.0,NaN,NaN,https://www.imdb.com/title/tt0076276/,Top Rated|Old Movies,NaN,NaN,NaN
2,Cannibal Holocaust,Adventure|Horror,An anthropologist ventures into the Amazon rai...,"rape,animal cruelty,gang rape,actual animal ki...",1985-06-21,95,Unrated,Ruggero Deodato,Robert Kerman,Francesca Ciardi,...,685.0,208.0,66800.0,$477,"$100,000 (estimated)",https://www.imdb.com/title/tt0078935/,Old Movies,$,477.0,100000.0
3,Saturn 3,Adventure|Horror|Sci-Fi,Two lovers stationed at a remote base in the a...,"psychopath,infatuation,robot sci fi,male rear ...",1980-02-15,88,R,Stanley Donen,Farrah Fawcett,Kirk Douglas,...,165.0,63.0,9300.0,"$9,000,000","£10,000,000 (estimated)",https://www.imdb.com/title/tt0079285/,Old Movies,£,9000000.0,12500000.0
4,Moskva slezam ne verit,Comedy|Drama|Romance,Three young provincial women come to Moscow in...,"city name in title,group of friends,soviet uni...",1980-02-11,150,PG,Vladimir Menshov,Vera Alentova,Aleksey Batalov,...,56.0,13.0,14600.0,$217,"$900,000 (estimated)",https://www.imdb.com/title/tt0079579/,Old Movies,$,217.0,900000.0


In [3]:
df.shape

(5637, 26)

In [4]:
df.dtypes

title               object
genres              object
description         object
keywords            object
release_date        object
duration             int64
content_rating      object
director            object
star_1              object
star_2              object
star_3              object
countries           object
languages           object
rating             float64
metascore          float64
votes                int64
user_reviews       float64
critic_reviews     float64
watchlist_count    float64
gross               object
budget              object
movie_url           object
category            object
currency_type       object
gross_usd          float64
budget_usd         float64
dtype: object

In [5]:
df.isna().sum()

title                 0
genres                0
description           0
keywords            164
release_date          0
duration              0
content_rating        0
director              0
star_1                0
star_2                0
star_3                0
countries             0
languages             0
rating                0
metascore          1988
votes                 0
user_reviews          0
critic_reviews        0
watchlist_count       0
gross              1390
budget             1784
movie_url             0
category              0
currency_type      1784
gross_usd          1390
budget_usd         1787
dtype: int64

---

# Data Preprocessing:

<span style='font-size:18px;'>
    ➤ Unknown values in columns is replaced with empty strings <br>
    ➤ Imputing metascore with the average of the median values of its genre’s metascore
</span>

In [6]:
df['keywords'] = df['keywords'].fillna('')

In [7]:
df = df[df['genres'] != 'Unknown'].reset_index(drop=True)

In [8]:
df_genre = df.copy()
df_genre['genres'] = df['genres'].str.split("|")
df_genre = df_genre.explode('genres')

In [9]:
genre_metascore = df_genre.groupby('genres')['metascore'].median().astype(int)
genre_metascore

genres
Action         53
Adventure      58
Animation      66
Biography      71
Comedy         57
Crime          60
Documentary    79
Drama          66
Family         57
Fantasy        51
History        73
Horror         54
Music          66
Musical        61
Mystery        59
News           67
Romance        57
Sci-Fi         57
Sport          62
Thriller       58
War            68
Western        64
Name: metascore, dtype: int64

In [10]:
def calculate_avg_metascore(x):
    genres_list = str(x).split('|')
    scores = []
    
    for g in genres_list:
        if g in genre_metascore:
            scores.append(genre_metascore[g])
    
    # If no valid genre impute global median
    if len(scores) == 0:
        return df['metascore'].median()
    
    return sum(scores) / len(scores)

In [11]:
df['metascore_imputed'] = df['metascore'].copy()

mask = df['metascore'].isna()

df.loc[mask, 'metascore_imputed'] = df.loc[mask, 'genres'].apply(calculate_avg_metascore)

In [12]:
df['metascore'] = df['metascore_imputed']
df.drop(columns=['metascore_imputed'], inplace=True)

In [13]:
df.isna().sum()

title                 0
genres                0
description           0
keywords              0
release_date          0
duration              0
content_rating        0
director              0
star_1                0
star_2                0
star_3                0
countries             0
languages             0
rating                0
metascore             0
votes                 0
user_reviews          0
critic_reviews        0
watchlist_count       0
gross              1384
budget             1777
movie_url             0
category              0
currency_type      1777
gross_usd          1384
budget_usd         1780
dtype: int64

In [14]:
for col in ['director', 'star_1', 'star_2', 'star_3', 'content_rating', 'languages']:
    df[col] = df[col].str.replace('Unknown','')

In [15]:
df['genres'] = df['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
df['countries'] = df['countries'].apply(lambda x: x.split('|'))
df['languages'] = df['languages'].apply(lambda x: x.split('|'))

---

# Text Vectorization:

<span style='font-size:18px;'>
TF-IDF converts text into numerical form while giving higher importance to unique words and reducing the effect of common words, improving similarity detection between movies. <br> <br>
Before applying TF-IDF, first combine all the features to make a text from these columns: <br>
&emsp;➤ description, keywords, genres, director, stars, content rating
</span>

In [16]:
df['combined_text'] = (
    df['description'] + " " +
    df['keywords'] + " " +
    df['genres'].apply(lambda x: " ".join(x)) + " " +
    df['director']*2 + " " +   
    df['star_1']*2 + " " +
    df['star_2'] + " " +
    df['star_3'] + " " +
    df['content_rating']
)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2)
)


tfidf_matrix = tfidf.fit_transform(df['combined_text'])

---

# MultiLabel Encoding:

<span style='font-size:18px;'>
Movies can belong to multiple genres. MultiLabel encoding allows us to represent multiple categories without losing information, unlike Label Encoding. Used it to encode: <br>
&emsp;➤ Genres, Countries, Languages
</span>

In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(df['genres'])
country_matrix = mlb.fit_transform(df['countries'])
language_matrix = mlb.fit_transform(df['languages'])

<span style='font-size:18px;'>
    ➤ Standard scaler is used to encode numerical features
</span>

In [19]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    'rating', 'votes', 'watchlist_count',
    'user_reviews', 'critic_reviews', 'metascore'
]

scaler = StandardScaler()
num_matrix = scaler.fit_transform(df[num_cols])

# Recommendation Model

<span style='font-size:18px;'>
    ➤ Genre Filtering
</span>

In [20]:
def filter_by_genre(movie_idx):
    
    movie_genres = set(df.loc[movie_idx, 'genres'])
    
    return df[df['genres'].apply(
        lambda x: len(set(x) & movie_genres) >= 2   
    )].index

<span style='font-size:18px;'>
    ➤ Language Filtering
</span>

In [21]:
def filter_by_language(movie_idx, indices):
    
    movie_lang = df.loc[movie_idx, 'languages']
    
    return [i for i in indices if movie_lang in df.loc[i, 'languages']]

<span style='font-size:18px;'> ➤ Feature Combining & Weighting using `hstack`
</span>

In [22]:
from scipy.sparse import hstack

final_matrix = hstack([
    tfidf_matrix * 0.6,     
    genre_matrix * 0.3,    
    country_matrix * 0.02,
    language_matrix * 0.03,
    num_matrix * 0.05       
])

<span style='font-size:18px;'> ➤ Siilarity Calculation using `Cosine Similarity`
</span>

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(final_matrix)

<span style='font-size:18px;'>
    ➤ Recommendation
</span>

In [24]:
def recommend(movie_name, top_n=5): 
    
    if movie_name not in df['title'].values:
        return f"Movie '{movie_name}' not found in dataset"
    idx = df[df['title'] == movie_name].index[0]

    filtered_indices = filter_by_genre(idx)
    filtered_indices = filter_by_language(idx, filtered_indices)
    
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]
    result = df.iloc[movie_indices][['title', 'genres', 'director', 'rating', 'metascore', 'movie_url']]
    
    return result

In [25]:
recommend('World War Z',10)

,title,genres,director,rating,metascore,movie_url
5069,Busanhaeng,"[Action, Horror, Thriller]",Yeon Sang-ho,7.6,73.0,https://www.imdb.com/title/tt5700672/
491,Aliens,"[Action, Adventure, Horror]",James Cameron,8.4,84.0,https://www.imdb.com/title/tt0090605/
2750,Prey,"[Action, Adventure, Horror]",Dan Trachtenberg,7.1,71.0,https://www.imdb.com/title/tt11866324/
1389,House of the Dead,"[Action, Adventure, Horror]",Uwe Boll,2.1,15.0,https://www.imdb.com/title/tt0317676/
4884,The Meg,"[Action, Adventure, Horror]",Jon Turteltaub,5.7,46.0,https://www.imdb.com/title/tt4779682/
5105,Escape Room,"[Action, Adventure, Horror]",Adam Robitel,6.4,48.0,https://www.imdb.com/title/tt5886046/
4958,Morbius,"[Action, Adventure, Horror]",Daniel Espinosa,5.1,35.0,https://www.imdb.com/title/tt5108870/
673,Predator,"[Action, Adventure, Horror]",John McTiernan,7.8,47.0,https://www.imdb.com/title/tt0093773/
5087,Underwater,"[Action, Adventure, Horror]",William Eubank,5.9,48.0,https://www.imdb.com/title/tt5774060/
1587,AVP: Alien vs. Predator,"[Action, Adventure, Horror]",Paul W.S. Anderson,5.7,29.0,https://www.imdb.com/title/tt0370263/


<span style='font-size:18px;'>
    ➤ Creating & Saving Recommendation CSV for Power BI Dashboard
</span>

In [26]:
def get_recommendation_df(movie_url, df, cosine_sim, top_n=5):

    if movie_url not in df['movie_url'].values:
        return pd.DataFrame()

    idx_list = df[df['movie_url'] == movie_url].index

    if len(idx_list) == 0:
        return pd.DataFrame()

    idx = idx_list[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    rows = []

    for i, score in sim_scores:
        rows.append({
            'input_movie': df.loc[idx, 'title'],
            'input_genres': df.loc[idx,'genres'],
            'input_url': movie_url,
            'recommended_movie': df.loc[i, 'title'],
            'recommended_url': df.loc[i, 'movie_url'],
            'recommended_genres': df.loc[i,'genres'],
            'similarity_score': round(score, 3)
        })

    return pd.DataFrame(rows)

In [27]:
all_recommendations = []

for movie in df['movie_url']:  # sample 50 movies
    rec_df = get_recommendation_df(movie, df, similarity)
    all_recommendations.append(rec_df)

final_rec_df = pd.concat(all_recommendations, ignore_index=True)

final_rec_df['input_genres'] = final_rec_df['input_genres'].str.join('|')
final_rec_df['recommended_genres'] = final_rec_df['recommended_genres'].str.join('|')

final_rec_df.head()

,input_movie,input_genres,input_url,recommended_movie,recommended_url,recommended_genres,similarity_score
0,Kate & Leopold,Comedy|Fantasy|Romance,https://www.imdb.com/title/tt0035423/,Midnight in Paris,https://www.imdb.com/title/tt1605783/,Comedy|Fantasy|Romance,0.532
1,Kate & Leopold,Comedy|Fantasy|Romance,https://www.imdb.com/title/tt0035423/,What Women Want,https://www.imdb.com/title/tt0207201/,Comedy|Fantasy|Romance,0.514
2,Kate & Leopold,Comedy|Fantasy|Romance,https://www.imdb.com/title/tt0035423/,Splash,https://www.imdb.com/title/tt0088161/,Comedy|Fantasy|Romance,0.496
3,Kate & Leopold,Comedy|Fantasy|Romance,https://www.imdb.com/title/tt0035423/,Teen Wolf,https://www.imdb.com/title/tt0090142/,Comedy|Fantasy|Romance,0.494
4,Kate & Leopold,Comedy|Fantasy|Romance,https://www.imdb.com/title/tt0035423/,Penelope,https://www.imdb.com/title/tt0472160/,Comedy|Fantasy|Romance,0.493


In [28]:
final_rec_df.shape

(28150, 7)

In [29]:
final_rec_df.to_csv("data/recommendations.csv", index=False)

---